# Agent: recommend

Develop and test **`agentic_scd.agents.recommend.recommend_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    UP1["classifications"]:::faded --> F1
    UP2["impacts"]:::faded --> F2
    UP3["simulation"]:::faded --> F3
    subgraph F["recommend_node (batch)"]
        F1["distinct categories"] --> F2["ACTION_BY_CATEGORY per category"]
        F2 --> F3["frame summary with simulation numbers"]
    end
    F3 --> D["recommendation<br/>Recommendation(actions, summary)"]
    D --> DOWN["downstream: render / output guardrail (Phase 7)"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `classifications`, `impacts`, and `simulation`
- **Writes:** `recommendation` (`Recommendation`: actions, summary)
- **Fallback / degradation:** no categories → a single default action; missing `simulation` → zeros

**Phase 7** replaces this with RAG-grounded mitigation generation (cited precedent) plus the output guardrail, behind the same `recommend_node` signature.

## Is the DB up? (optional)

In [ ]:
# Optional: this agent runs fine offline on synthetic sample state. This snippet just
# reports whether the live DB is reachable (Setup section of 00_orchestration brings
# it up).
from agentic_scd.devtools import db_status

status = db_status()
print(status.detail)
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")

## Build a representative input state

In [ ]:
from agentic_scd.agents.classify import classify_node
from agentic_scd.agents.impact import impact_node
from agentic_scd.agents.simulate import simulate_node
from agentic_scd.devtools import sample_state

# recommend reads classifications + impacts + simulation: build the full upstream chain.
state = sample_state(count=2)
state.update(classify_node(state))
state.update(impact_node(state))
state.update(simulate_node(state))

## Call `recommend_node` in isolation

In [ ]:
from agentic_scd.agents.recommend import recommend_node

state.update(recommend_node(state))
rec = state["recommendation"]
for action in rec.actions:
    print("-", action)
print(f"\n({rec.summary})")

## Iterate here

This is your dev surface: tweak the input above, re-run, and watch `recommend_node`'s output change. When you deepen this agent in its phase, keep the node signature the same so the rest of the graph is unaffected.